<a href="https://colab.research.google.com/github/click2shivesh/shivesh-uta-aiml-py/blob/main/Shivesh_NLP_Medical_Diagnosis._RAG_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medical Diagnosis Support Using LLM and Retrieval-Augmented Generation

**Prepared by: Shivesh**

## Problem Statement

### Business Context

Healthcare professionals work with large volumes of clinical information while being expected to make timely and well-informed decisions. Searching a lengthy medical reference manually can be slow, particularly when the question involves symptoms, emergency precautions, treatment, and follow-up care.

A large language model can summarise information in natural language, but a model operating without a trusted source may produce incomplete or unsupported statements. Retrieval-Augmented Generation addresses this limitation by first locating relevant content in an approved knowledge source and then asking the language model to answer from that evidence.

### Objective

The objective of this project is to develop and assess a RAG-based medical knowledge assistant using *The Merck Manual of Diagnosis and Therapy*. The notebook compares:

1. a standalone Hugging Face language model;
2. the same model with prompt engineering;
3. a RAG pipeline grounded in the supplied medical manual; and
4. multiple RAG parameter settings evaluated for groundedness and relevance.

> This project is an educational decision-support prototype. It is not intended to diagnose a patient or replace a qualified healthcare professional.

## Data Description

The supplied document is a PDF edition of *The Merck Manual of Diagnosis and Therapy*. It contains more than 4,000 PDF pages organised into 23 medical sections. The project focuses on five questions covering critical care, gastrointestinal surgery, dermatology, brain injury, and fracture management.

## Questions to Answer

1. What is the protocol for managing sepsis in a critical care unit?
2. What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed?
3. What are the treatments and possible causes of sudden patchy hair loss?
4. What treatments are recommended for traumatic injury to brain tissue?
5. What precautions, treatment steps, and recovery considerations apply to a leg fracture during a hiking trip?

# Installing and Importing Libraries

Run this notebook in Google Colab with a **T4 GPU**:

`Runtime → Change runtime type → T4 GPU`

In [ ]:
!nvidia-smi

In [1]:
# Build llama-cpp-python with CUDA support for the Colab GPU.
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q --upgrade --force-reinstall --no-cache-dir llama-cpp-python==0.2.90

# Libraries used for PDF extraction, embeddings, vector search, and tabular summaries.
!pip install -q \
    huggingface-hub==0.34.4 \
    pymupdf==1.26.3 \
    langchain==0.3.27 \
    langchain-community==0.3.27 \
    langchain-text-splitters==0.3.9 \
    chromadb==1.0.20 \
    sentence-transformers==5.1.0 \
    tiktoken==0.11.0 \
    pandas==2.2.2 \
    numpy==1.26.4

print("Install step completed. If you see dependency warnings, restart the runtime and re-run this cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 MB 191.6 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve

### Installation Observation

The notebook uses a quantised GGUF model through `llama-cpp-python`. Quantisation allows a 7-billion-parameter instruction model to run within the memory available on a Colab T4 GPU. The remaining libraries support document extraction, semantic embeddings, persistent vector search, and result comparison.

In [ ]:
import os
import re
import gc
import json
import time
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

print("Imports completed successfully.")

In [ ]:
medical_questions = {
    "Q1": "What is the protocol for managing sepsis in a critical care unit?",
    "Q2": "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Q3": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "Q4": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "Q5": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
}

pd.DataFrame(
    [{"Question ID": key, "Question": value} for key, value in medical_questions.items()]
)

# 1. Question Answering Using a Standalone LLM

This section measures the behaviour of the language model before it is given any content from the Merck Manual. These responses provide a baseline for later comparison.

## Downloading and Loading the Hugging Face Model

In [ ]:
MODEL_REPOSITORY = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
MODEL_FILENAME = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

gguf_path = hf_hub_download(
    repo_id=MODEL_REPOSITORY,
    filename=MODEL_FILENAME
)

print("Downloaded model:", gguf_path)

In [ ]:
medical_llm = Llama(
    model_path=gguf_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_batch=512,
    seed=21,
    verbose=False
)

print("Mistral model loaded on the available accelerator.")

## Response Generation Function

In [ ]:
def make_instruction_prompt(role_text, task_text):
    """Create the instruction format expected by Mistral-Instruct."""
    return f"<s>[INST] {role_text.strip()}\n\n{task_text.strip()} [/INST]"


def ask_model(
    task,
    role="You are a careful medical information assistant.",
    max_new_tokens=300,
    temperature=0.0,
    top_p=0.9,
    top_k=40,
    repetition_penalty=1.1,
    random_seed=21
):
    """Run one controlled generation and return only the generated answer."""

    prompt_text = make_instruction_prompt(role, task)

    result = medical_llm(
        prompt=prompt_text,
        max_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repetition_penalty,
        seed=random_seed,
        stop=["</s>", "[INST]"]
    )

    return result["choices"][0]["text"].strip()

## Generating Baseline Answers

In [ ]:
baseline_records = []

for question_id, question_text in medical_questions.items():
    started = time.time()

    generated_answer = ask_model(
        task=question_text,
        role="You are a medical information assistant. Answer the question clearly.",
        max_new_tokens=320,
        temperature=0.0
    )

    baseline_records.append({
        "question_id": question_id,
        "question": question_text,
        "baseline_answer": generated_answer,
        "word_count": len(generated_answer.split()),
        "generation_time_seconds": round(time.time() - started, 2)
    })

    print("\n" + "=" * 95)
    print(question_id, question_text)
    print("-" * 95)
    print(generated_answer)

baseline_answers = pd.DataFrame(baseline_records)
baseline_answers

### Baseline LLM Observations

- The model can produce fluent answers without receiving any reference text.
- The clinical tone of an answer does not establish that every claim is correct or supported.
- Compound questions may be answered unevenly; for example, the model may describe symptoms but provide limited detail about precautions or recovery.
- A standalone response has no page citation or evidence trail, making verification difficult.
- The baseline is therefore useful for comparison but is not the preferred approach for a medical knowledge application.

# 2. Question Answering Using Prompt Engineering

Five prompt designs are tested. Each design changes both the instruction strategy and selected generation parameters. All five questions are run through every configuration so the comparison is consistent.

In [ ]:
prompt_setups = [
    {
        "name": "P1_Basic",
        "role": "You are a medical information assistant.",
        "task_prefix": "Answer the following question directly and concisely.",
        "temperature": 0.0,
        "top_p": 0.90,
        "top_k": 40,
        "max_new_tokens": 300
    },
    {
        "name": "P2_Clinical_Structure",
        "role": (
            "You support healthcare professionals with educational medical information. "
            "Use clear headings and separate presentation, assessment, treatment, and follow-up."
        ),
        "task_prefix": "Address every part of the question using a structured response.",
        "temperature": 0.1,
        "top_p": 0.90,
        "top_k": 30,
        "max_new_tokens": 380
    },
    {
        "name": "P3_Safety_Focused",
        "role": (
            "You are a conservative clinical information assistant. "
            "Do not invent doses. State urgent warning signs and when emergency care is required."
        ),
        "task_prefix": "Give a cautious answer that prioritises immediate safety.",
        "temperature": 0.0,
        "top_p": 0.82,
        "top_k": 20,
        "max_new_tokens": 360
    },
    {
        "name": "P4_Explain_Reasoning",
        "role": (
            "You are an evidence-oriented medical educator. Explain the condition, likely causes, "
            "main signs, management options, and limitations of the available information."
        ),
        "task_prefix": "Provide a concise explanation with numbered management steps.",
        "temperature": 0.2,
        "top_p": 0.92,
        "top_k": 50,
        "max_new_tokens": 420
    },
    {
        "name": "P5_Professional_Summary",
        "role": (
            "You are preparing a clinical knowledge summary. Use precise language, avoid speculation, "
            "and finish with a brief safety note."
        ),
        "task_prefix": "Produce a professional summary covering symptoms, treatment, precautions, and recovery when relevant.",
        "temperature": 0.1,
        "top_p": 0.88,
        "top_k": 35,
        "max_new_tokens": 430
    }
]

pd.DataFrame(prompt_setups)[
    ["name", "temperature", "top_p", "top_k", "max_new_tokens"]
]

In [ ]:
engineered_records = []

for setup in prompt_setups:
    for question_id, question_text in medical_questions.items():

        full_task = (
            f"{setup['task_prefix']}\n\n"
            f"Medical question: {question_text}"
        )

        answer_text = ask_model(
            task=full_task,
            role=setup["role"],
            max_new_tokens=setup["max_new_tokens"],
            temperature=setup["temperature"],
            top_p=setup["top_p"],
            top_k=setup["top_k"]
        )

        engineered_records.append({
            "prompt_design": setup["name"],
            "question_id": question_id,
            "answer": answer_text,
            "word_count": len(answer_text.split())
        })

        print("\n" + "=" * 95)
        print(setup["name"], "|", question_id)
        print("-" * 95)
        print(answer_text)

prompt_results = pd.DataFrame(engineered_records)
prompt_results

In [ ]:
prompt_comparison = (
    prompt_results
    .groupby("prompt_design", as_index=False)
    .agg(
        number_of_answers=("answer", "count"),
        average_words=("word_count", "mean"),
        shortest_answer=("word_count", "min"),
        longest_answer=("word_count", "max")
    )
    .round(1)
)

prompt_comparison

### Prompt Engineering Observations

- Role-based prompting improves the professional focus of the responses.
- A structured instruction makes it easier to verify whether symptoms, immediate actions, treatment, and follow-up have all been addressed.
- Safety-focused prompting reduces speculative wording but may produce a less detailed answer.
- Higher response limits support multi-part questions, although longer output also creates more claims that require verification.
- Prompt engineering improves form and consistency, but the model is still relying on pretrained knowledge rather than the supplied manual.

# 3. Data Preparation for Retrieval-Augmented Generation

## Loading the Medical Manual

In [ ]:
pdf_location = Path("/content/medical_diagnosis_manual.pdf")

if not pdf_location.exists():
    from google.colab import files

    print("Upload the supplied Merck Manual PDF.")
    uploaded_files = files.upload()

    pdf_names = [
        filename for filename in uploaded_files
        if filename.lower().endswith(".pdf")
    ]

    if not pdf_names:
        raise FileNotFoundError("A PDF was not uploaded.")

    pdf_location = Path("/content") / pdf_names[0]

manual_loader = PyMuPDFLoader(str(pdf_location))
page_documents = manual_loader.load()

print("PDF used:", pdf_location)
print("Number of extracted PDF pages:", len(page_documents))

## Initial Data Inspection

In [ ]:
for page_index, document in enumerate(page_documents[:5], start=1):
    print("\n" + "=" * 80)
    print(f"PDF page {page_index}")
    print("-" * 80)
    preview = document.page_content.strip()
    print(preview[:650] if preview else "[No extractable text]")

In [ ]:
page_character_counts = [
    len(document.page_content.strip())
    for document in page_documents
]

pd.DataFrame({
    "measure": [
        "Total PDF pages",
        "Pages with fewer than 50 extracted characters",
        "Median characters per page",
        "Largest extracted page"
    ],
    "value": [
        len(page_documents),
        sum(count < 50 for count in page_character_counts),
        int(np.median(page_character_counts)),
        max(page_character_counts)
    ]
})

### Data Inspection Observation

The PDF contains enough text for semantic retrieval, but repeated ownership notices and page-level boilerplate are present throughout the document. Such repeated strings do not add medical meaning and can distort embeddings, so they are removed before chunking.

## Cleaning Extracted Text

In [ ]:
repeated_text_patterns = [
    r"click2shivesh@gmail\.com",
    r"WPFSN4D3ZX",
    r"This file is meant for personal use by .*? only\.",
    r"Sharing or publishing the contents in part or full is liable for legal action\."
]


def normalise_manual_text(raw_text):
    \"\"\"Remove repeated ownership notices and normalise spacing.\"\"\"

    cleaned_text = raw_text

    for pattern in repeated_text_patterns:
        cleaned_text = re.sub(
            pattern,
            " ",
            cleaned_text,
            flags=re.IGNORECASE
        )

    cleaned_text = re.sub(r"\\s+", " ", cleaned_text).strip()
    return cleaned_text


usable_pages = []

for document in page_documents:
    revised_text = normalise_manual_text(document.page_content)

    if len(revised_text) >= 50:
        document.page_content = revised_text
        document.metadata["pdf_page"] = int(
            document.metadata.get("page", 0)
        ) + 1
        usable_pages.append(document)

print("Usable pages after cleaning:", len(usable_pages))
print("\\nCleaned sample:\\n", usable_pages[0].page_content[:600])

## Splitting the Manual into Chunks

In [ ]:
default_chunk_size = 900
default_overlap = 180

document_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=default_chunk_size,
    chunk_overlap=default_overlap,
    separators=["\n\n", "\n", ". ", " ", ""]
)

manual_chunks = document_splitter.split_documents(usable_pages)

for chunk_number, chunk in enumerate(manual_chunks):
    chunk.metadata["chunk_number"] = chunk_number

print("Total chunks created:", len(manual_chunks))
print("First chunk metadata:", manual_chunks[0].metadata)
print("\nFirst chunk preview:\n", manual_chunks[0].page_content[:650])

### Chunking Observation

A 900-token chunk with 180-token overlap is used as the starting point. This size is large enough to retain nearby symptoms and treatment statements while remaining small enough for targeted retrieval. The overlap protects against losing meaning where a section crosses a chunk boundary.

## Loading the Embedding Model

In [ ]:
text_embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

embedding_check = text_embedder.embed_query(
    "critical care management of sepsis"
)

print("Embedding vector length:", len(embedding_check))
print("Sample values:", embedding_check[:8])

### Embedding Observation

The selected sentence-transformer creates a 384-dimensional semantic representation for each query and chunk. Normalised embeddings allow the vector database to compare content by meaning rather than relying only on exact keyword matches.

## Creating the Vector Database

In [ ]:
base_store_directory = "/content/shivesh_merck_chroma"

if Path(base_store_directory).exists() and any(Path(base_store_directory).iterdir()):
    medical_store = Chroma(
        collection_name="merck_manual_default",
        persist_directory=base_store_directory,
        embedding_function=text_embedder
    )
    print("Existing vector database loaded.")
else:
    medical_store = Chroma.from_documents(
        documents=manual_chunks,
        embedding=text_embedder,
        collection_name="merck_manual_default",
        persist_directory=base_store_directory
    )
    print("New vector database created.")

print("Vector index is ready.")

## Defining the Retriever

In [ ]:
medical_retriever = medical_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.65
    }
)

retrieval_test = medical_retriever.invoke(
    "sepsis and septic shock treatment in an intensive care unit"
)

for rank, document in enumerate(retrieval_test, start=1):
    print(
        f"Rank {rank} | PDF page {document.metadata.get('pdf_page')} | "
        f"Chunk {document.metadata.get('chunk_number')}"
    )
    print(document.page_content[:420])
    print()

### Retriever Observation

Maximum Marginal Relevance is used because it attempts to retrieve both relevant and non-duplicative passages. This is helpful for medical questions that require several aspects of care rather than a single definition.

# 4. Question Answering Using RAG

## RAG Prompt and Supporting Functions

In [ ]:
rag_role = """
You are an educational medical knowledge assistant.

Use only the supplied excerpts from The Merck Manual.

Requirements:
- Do not introduce a treatment, medicine, dose, cause, or timing that is absent from the excerpts.
- Address every component of the question.
- Separate immediate precautions, assessment, treatment, and recovery when applicable.
- State clearly when the excerpts do not contain enough information.
- Cite relevant PDF pages in square brackets.
- Do not present the answer as a substitute for professional clinical judgement.
""".strip()


def join_retrieved_context(documents, token_limit=2600):
    """Combine retrieved chunks while respecting a context-token budget."""

    encoder = tiktoken.get_encoding("cl100k_base")
    assembled_blocks = []
    token_total = 0

    for rank, document in enumerate(documents, start=1):
        pdf_page = document.metadata.get("pdf_page", "unknown")
        chunk_number = document.metadata.get("chunk_number", "unknown")

        source_block = (
            f"[Source {rank}; PDF page {pdf_page}; chunk {chunk_number}]\n"
            f"{document.page_content.strip()}"
        )

        encoded_block = encoder.encode(source_block)
        available_tokens = token_limit - token_total

        if available_tokens <= 0:
            break

        if len(encoded_block) > available_tokens:
            assembled_blocks.append(
                encoder.decode(encoded_block[:available_tokens])
            )
            break

        assembled_blocks.append(source_block)
        token_total += len(encoded_block)

    return "\n\n-----\n\n".join(assembled_blocks)


def answer_with_manual(
    question,
    retriever=medical_retriever,
    retrieval_k=5,
    context_token_limit=2600,
    answer_token_limit=440,
    temperature=0.0,
    top_p=0.9,
    top_k=30
):
    """Retrieve manual evidence and generate one page-cited answer."""

    retrieved_documents = retriever.invoke(question)[:retrieval_k]
    context_text = join_retrieved_context(
        retrieved_documents,
        token_limit=context_token_limit
    )

    rag_task = f"""
MERCK MANUAL EXCERPTS:
{context_text}

QUESTION:
{question}

Prepare a concise, structured answer that is fully supported by the excerpts.
""".strip()

    generated_text = ask_model(
        task=rag_task,
        role=rag_role,
        max_new_tokens=answer_token_limit,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    cited_pages = sorted({
        document.metadata.get("pdf_page")
        for document in retrieved_documents
        if document.metadata.get("pdf_page") is not None
    })

    return {
        "answer": generated_text,
        "context": context_text,
        "pages": cited_pages,
        "documents": retrieved_documents
    }

## RAG Answers for the Five Questions

In [ ]:
initial_rag_records = []
initial_rag_details = {}

for question_id, question_text in medical_questions.items():

    rag_result = answer_with_manual(
        question=question_text,
        retrieval_k=5,
        answer_token_limit=440,
        temperature=0.0
    )

    initial_rag_details[question_id] = rag_result

    initial_rag_records.append({
        "question_id": question_id,
        "question": question_text,
        "rag_answer": rag_result["answer"],
        "retrieved_pdf_pages": rag_result["pages"],
        "word_count": len(rag_result["answer"].split())
    })

    print("\n" + "=" * 95)
    print(question_id, "| Retrieved pages:", rag_result["pages"])
    print("-" * 95)
    print(rag_result["answer"])

initial_rag_answers = pd.DataFrame(initial_rag_records)
initial_rag_answers

### Initial RAG Observations

- The answers now have an evidence trail through retrieved PDF pages.
- Restricting the model to supplied excerpts reduces reliance on hidden pretrained knowledge.
- Retrieval quality directly affects answer quality. When a relevant chapter is missed, the answer may remain incomplete even when generation is well controlled.
- The instruction to declare insufficient context is preferable to filling gaps with unsupported information.

# 5. RAG Fine-Tuning Experiments

Five configurations are compared. Unlike prompt-only tuning, these experiments vary document chunking, retrieval strategy, number of retrieved chunks, and LLM generation parameters.

In [ ]:
rag_configurations = [
    {
        "configuration": "R1_Focused_Similarity",
        "chunk_size": 550,
        "overlap": 90,
        "search_type": "similarity",
        "k": 3,
        "fetch_k": None,
        "lambda_mult": None,
        "temperature": 0.0,
        "top_p": 0.88,
        "top_k": 25,
        "answer_tokens": 360
    },
    {
        "configuration": "R2_Medium_Similarity",
        "chunk_size": 750,
        "overlap": 140,
        "search_type": "similarity",
        "k": 5,
        "fetch_k": None,
        "lambda_mult": None,
        "temperature": 0.1,
        "top_p": 0.90,
        "top_k": 35,
        "answer_tokens": 400
    },
    {
        "configuration": "R3_Balanced_MMR",
        "chunk_size": 900,
        "overlap": 180,
        "search_type": "mmr",
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.65,
        "temperature": 0.0,
        "top_p": 0.90,
        "top_k": 30,
        "answer_tokens": 440
    },
    {
        "configuration": "R4_Broad_MMR",
        "chunk_size": 1150,
        "overlap": 230,
        "search_type": "mmr",
        "k": 6,
        "fetch_k": 24,
        "lambda_mult": 0.55,
        "temperature": 0.1,
        "top_p": 0.92,
        "top_k": 40,
        "answer_tokens": 470
    },
    {
        "configuration": "R5_Large_Context_MMR",
        "chunk_size": 1450,
        "overlap": 280,
        "search_type": "mmr",
        "k": 8,
        "fetch_k": 32,
        "lambda_mult": 0.50,
        "temperature": 0.0,
        "top_p": 0.85,
        "top_k": 20,
        "answer_tokens": 500
    }
]

pd.DataFrame(rag_configurations)

In [ ]:
def create_experiment_retriever(configuration):
    """Build a temporary vector index for one chunking/retrieval configuration."""

    experiment_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        chunk_size=configuration["chunk_size"],
        chunk_overlap=configuration["overlap"],
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    experiment_chunks = experiment_splitter.split_documents(usable_pages)

    for position, chunk in enumerate(experiment_chunks):
        chunk.metadata["chunk_number"] = position

    safe_name = re.sub(
        r"[^a-z0-9_]",
        "_",
        configuration["configuration"].lower()
    )
    experiment_directory = f"/content/{safe_name}_chroma"

    if Path(experiment_directory).exists() and any(Path(experiment_directory).iterdir()):
        experiment_store = Chroma(
            collection_name=safe_name,
            persist_directory=experiment_directory,
            embedding_function=text_embedder
        )
    else:
        experiment_store = Chroma.from_documents(
            documents=experiment_chunks,
            embedding=text_embedder,
            collection_name=safe_name,
            persist_directory=experiment_directory
        )

    if configuration["search_type"] == "mmr":
        experiment_retriever = experiment_store.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": configuration["k"],
                "fetch_k": configuration["fetch_k"],
                "lambda_mult": configuration["lambda_mult"]
            }
        )
    else:
        experiment_retriever = experiment_store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": configuration["k"]}
        )

    return experiment_chunks, experiment_store, experiment_retriever

In [ ]:
rag_experiment_records = []
rag_experiment_details = {}

for configuration in rag_configurations:

    print("\n" + "#" * 100)
    print("Building:", configuration["configuration"])

    temporary_chunks, temporary_store, temporary_retriever = (
        create_experiment_retriever(configuration)
    )

    print("Chunks in this configuration:", len(temporary_chunks))

    for question_id, question_text in medical_questions.items():

        experiment_result = answer_with_manual(
            question=question_text,
            retriever=temporary_retriever,
            retrieval_k=configuration["k"],
            context_token_limit=2700,
            answer_token_limit=configuration["answer_tokens"],
            temperature=configuration["temperature"],
            top_p=configuration["top_p"],
            top_k=configuration["top_k"]
        )

        rag_experiment_details[
            (configuration["configuration"], question_id)
        ] = experiment_result

        rag_experiment_records.append({
            "configuration": configuration["configuration"],
            "question_id": question_id,
            "chunk_size": configuration["chunk_size"],
            "overlap": configuration["overlap"],
            "search_type": configuration["search_type"],
            "k": configuration["k"],
            "temperature": configuration["temperature"],
            "retrieved_pages": experiment_result["pages"],
            "answer": experiment_result["answer"],
            "word_count": len(experiment_result["answer"].split())
        })

        print("\n", configuration["configuration"], "|", question_id)
        print("Pages:", experiment_result["pages"])
        print(experiment_result["answer"])

    del temporary_chunks, temporary_store, temporary_retriever
    gc.collect()

rag_experiment_results = pd.DataFrame(rag_experiment_records)
rag_experiment_results

In [ ]:
rag_configuration_summary = (
    rag_experiment_results
    .groupby(
        [
            "configuration",
            "chunk_size",
            "overlap",
            "search_type",
            "k",
            "temperature"
        ],
        as_index=False
    )
    .agg(
        responses=("answer", "count"),
        average_answer_words=("word_count", "mean"),
        average_number_of_pages=(
            "retrieved_pages",
            lambda page_lists: np.mean([len(pages) for pages in page_lists])
        )
    )
    .round(2)
)

rag_configuration_summary

### RAG Tuning Observations

- Small chunks produce focused retrieval but can separate related clinical instructions.
- Medium-sized chunks generally preserve more context without overwhelming the prompt.
- Similarity search may return several near-duplicate passages, whereas MMR tends to provide broader coverage.
- Increasing `k` can help multi-part questions, but a very high value may add unrelated text.
- Temperature values close to zero are suitable for evidence summarisation because creativity is not the objective.
- The final configuration should be selected after considering source relevance, groundedness, completeness, and clarity together.

# 6. Selecting the Final RAG Configuration

`R3_Balanced_MMR` is used as the provisional final configuration because it combines moderate chunk size, overlap, diversified retrieval, and deterministic generation. The evaluation section will confirm whether this choice performs well.

In [ ]:
selected_configuration_name = "R3_Balanced_MMR"

selected_configuration = next(
    item for item in rag_configurations
    if item["configuration"] == selected_configuration_name
)

final_chunks, final_store, final_retriever = create_experiment_retriever(
    selected_configuration
)

pd.DataFrame([selected_configuration])

In [ ]:
final_answer_records = []
final_answer_details = {}

for question_id, question_text in medical_questions.items():

    final_result = answer_with_manual(
        question=question_text,
        retriever=final_retriever,
        retrieval_k=selected_configuration["k"],
        context_token_limit=2700,
        answer_token_limit=selected_configuration["answer_tokens"],
        temperature=selected_configuration["temperature"],
        top_p=selected_configuration["top_p"],
        top_k=selected_configuration["top_k"]
    )

    final_answer_details[question_id] = final_result

    final_answer_records.append({
        "question_id": question_id,
        "question": question_text,
        "final_answer": final_result["answer"],
        "source_pages": final_result["pages"]
    })

    print("\n" + "=" * 95)
    print("FINAL", question_id, "| Source pages:", final_result["pages"])
    print("-" * 95)
    print(final_result["answer"])

final_rag_answers = pd.DataFrame(final_answer_records)
final_rag_answers

# 7. Output Evaluation

The final responses are evaluated separately for:

- **Groundedness:** Are the answer’s important statements supported by the retrieved excerpts?
- **Relevance:** Does the answer directly and completely address the question?

The evaluator is asked to return JSON so the results can be collected consistently.

In [ ]:
groundedness_evaluator_role = """
You are evaluating a medical answer for groundedness.

Judge the answer only against the supplied excerpts. Do not reward a statement
merely because it sounds medically plausible.

Scoring:
1 = largely unsupported or contradictory
2 = several important claims are unsupported
3 = partly supported, with material gaps
4 = strongly supported, with minor unsupported detail
5 = fully supported by the supplied excerpts

Return valid JSON only:
{
  "score": 1,
  "reason": "brief explanation",
  "unsupported_claims": []
}
""".strip()


relevance_evaluator_role = """
You are evaluating whether a medical answer is relevant and complete for the question.

Scoring:
1 = irrelevant
2 = addresses only a small part
3 = relevant but materially incomplete
4 = mostly complete and direct
5 = complete, direct, and clearly organised

Return valid JSON only:
{
  "score": 1,
  "reason": "brief explanation",
  "missing_points": []
}
""".strip()

In [ ]:
def extract_json_object(raw_output):
    \"\"\"Extract the first JSON object returned by the evaluator.\"\"\"

    json_match = re.search(r"\{.*\}", raw_output, flags=re.DOTALL)

    if json_match is None:
        return {
            "score": None,
            "reason": raw_output,
            "issues": []
        }

    try:
        return json.loads(json_match.group(0))
    except json.JSONDecodeError:
        return {
            "score": None,
            "reason": raw_output,
            "issues": []
        }


def score_groundedness(context_text, answer_text):
    evaluation_task = f\"\"\"
REFERENCE EXCERPTS:
{context_text}

ANSWER TO EVALUATE:
{answer_text}
\"\"\".strip()

    raw_evaluation = ask_model(
        task=evaluation_task,
        role=groundedness_evaluator_role,
        max_new_tokens=220,
        temperature=0.0,
        top_p=0.8,
        top_k=20
    )

    parsed = extract_json_object(raw_evaluation)
    parsed["raw_output"] = raw_evaluation
    return parsed


def score_relevance(question_text, answer_text):
    evaluation_task = f\"\"\"
QUESTION:
{question_text}

ANSWER TO EVALUATE:
{answer_text}
\"\"\".strip()

    raw_evaluation = ask_model(
        task=evaluation_task,
        role=relevance_evaluator_role,
        max_new_tokens=220,
        temperature=0.0,
        top_p=0.8,
        top_k=20
    )

    parsed = extract_json_object(raw_evaluation)
    parsed["raw_output"] = raw_evaluation
    return parsed

In [ ]:
evaluation_records = []

for question_id, question_text in medical_questions.items():

    answer_detail = final_answer_details[question_id]

    groundedness_result = score_groundedness(
        context_text=answer_detail["context"],
        answer_text=answer_detail["answer"]
    )

    relevance_result = score_relevance(
        question_text=question_text,
        answer_text=answer_detail["answer"]
    )

    evaluation_records.append({
        "question_id": question_id,
        "groundedness_score": groundedness_result.get("score"),
        "groundedness_reason": groundedness_result.get("reason"),
        "unsupported_claims": groundedness_result.get(
            "unsupported_claims", []
        ),
        "relevance_score": relevance_result.get("score"),
        "relevance_reason": relevance_result.get("reason"),
        "missing_points": relevance_result.get("missing_points", [])
    })

evaluation_results = pd.DataFrame(evaluation_records)
evaluation_results

In [ ]:
evaluation_overview = pd.DataFrame({
    "evaluation_measure": [
        "Mean groundedness score",
        "Mean relevance score",
        "Number of final responses evaluated"
    ],
    "result": [
        pd.to_numeric(
            evaluation_results["groundedness_score"],
            errors="coerce"
        ).mean(),
        pd.to_numeric(
            evaluation_results["relevance_score"],
            errors="coerce"
        ).mean(),
        len(evaluation_results)
    ]
})

evaluation_overview

### Evaluation Observations

- Groundedness and relevance measure different weaknesses. A response can be relevant but unsupported, or supported but incomplete.
- Unsupported-claim output identifies where the generation prompt needs stricter control.
- Missing-point output helps determine whether more or different chunks should be retrieved.
- The automated evaluator provides a consistent prototype metric, but it uses the same underlying model and therefore cannot replace independent review by medical experts.

# 8. Overall Comparison

In [ ]:
approach_comparison = pd.DataFrame({
    "Approach": [
        "Standalone LLM",
        "Prompt-engineered LLM",
        "RAG-grounded LLM"
    ],
    "Main benefit": [
        "Fast and fluent baseline response",
        "Improved organisation and caution",
        "Evidence-backed answer with source pages"
    ],
    "Main limitation": [
        "No supplied evidence or traceability",
        "Prompting cannot guarantee factual support",
        "Performance depends on extraction and retrieval quality"
    ],
    "Recommended use": [
        "Baseline experimentation only",
        "Prompt design and response formatting",
        "Preferred method for the knowledge-assistant prototype"
    ]
})

approach_comparison

# 9. Actionable Insights and Business Recommendations

## Key Insights

### Faster information access

The RAG pipeline can locate relevant sections of a multi-thousand-page manual and summarise them in a usable format. This can reduce time spent manually navigating a large reference.

### Better traceability

Providing PDF pages alongside the answer allows a healthcare professional to inspect the source and verify the summary.

### More consistent knowledge support

A centrally maintained knowledge base can provide teams with access to the same approved reference material and response rules.

### Human review remains essential

The system should support clinical work rather than make autonomous diagnoses or treatment decisions. Individual care still requires history, examination, investigations, contraindication checks, and professional judgement.

## Business Recommendations

1. Begin with a limited educational pilot using non-identifiable questions.
2. Display retrieved passages and document versions with every answer.
3. Create an escalation response when supporting evidence is missing.
4. Review retrieval errors and unsupported claims regularly.
5. Use role-based access, audit logs, and privacy controls.
6. Validate responses with clinicians before wider use.
7. Replace the demonstration source with current, approved medical guidance before production deployment.

## Important Source Limitation

The supplied manual is the 19th edition published in 2011. A response can be perfectly grounded in that PDF and still be outdated relative to current clinical practice. Production use would therefore require current manuals, institutional protocols, drug references, and scheduled content updates.

# 10. Limitations and Future Improvements

## Current Limitations

- PDF extraction may lose tables, images, and layout relationships.
- Repeated headers and unusual characters may remain after cleaning.
- A dense retriever can miss an exact medical term or select a neighbouring topic.
- The language model can misread an otherwise relevant excerpt.
- The evaluator is model-based rather than clinician-based.
- The system has no patient-specific data and must not be used for individual diagnosis.

## Future Improvements

- combine semantic retrieval with keyword search;
- add a cross-encoder reranker;
- use section and chapter metadata;
- perform layout-aware extraction of tables and figures;
- attach citations to individual claims;
- introduce confidence thresholds and abstention;
- evaluate against clinician-written reference answers;
- test with current medical sources; and
- monitor retrieval and generation quality after every update.

# 11. Conclusion

The notebook demonstrates that prompt engineering can improve the structure and caution of standalone LLM responses, but it cannot establish that the content comes from the supplied medical reference.

The RAG approach provides a stronger solution by retrieving relevant Merck Manual excerpts before generation and exposing source pages for verification. Parameter experiments also show that chunk size, overlap, retrieval method, `k`, and generation settings influence completeness and noise.

The most appropriate role for the prototype is a **medical knowledge retrieval and summarisation assistant under professional supervision**. It should not be presented as an autonomous diagnosis system.

# Saving Project Results

In [ ]:
results_folder = Path("/content/shivesh_medical_rag_outputs")
results_folder.mkdir(parents=True, exist_ok=True)

baseline_answers.to_csv(
    results_folder / "standalone_llm_answers.csv",
    index=False
)

prompt_results.to_csv(
    results_folder / "prompt_engineering_results.csv",
    index=False
)

initial_rag_answers.to_csv(
    results_folder / "initial_rag_answers.csv",
    index=False
)

rag_experiment_results.to_csv(
    results_folder / "rag_parameter_experiments.csv",
    index=False
)

final_rag_answers.to_csv(
    results_folder / "final_rag_answers.csv",
    index=False
)

evaluation_results.to_csv(
    results_folder / "groundedness_relevance_evaluation.csv",
    index=False
)

print("Files saved in:", results_folder)
for saved_file in sorted(results_folder.iterdir()):
    print("-", saved_file.name)